# Notebook 03 — Transformations métier
## Projet MSPR · Electio-Analytics · Pays de la Loire

---

### 🎯 Objectif de ce notebook

Ce notebook correspond à la **troisième étape du pipeline ETL** : les **transformations métier**.

Il prend en entrée les fichiers standardisés (`stg_std_*.csv`) et produit :

- `tr_elections_region.csv` → participation et résultats agrégés **au niveau région PDL**
- `tr_indicateurs_region.csv` → indicateurs socio-éco agrégés **au niveau région PDL**
- `tr_deltas_indicateurs.csv` → **évolution (delta)** de chaque indicateur entre 2012→2017→2022
- `tr_correlations.csv` → **corrélations** entre chaque indicateur et le taux de participation

### 📋 Règles de transformation appliquées (consignes du prof)

| Règle | Application |
|---|---|
| **Filtre par région** | On filtre sur les 5 depts PDL avant toute agrégation |
| **NaN → 0** | Toutes les valeurs manquantes sont remplacées par 0 |
| **Moyenne sur la colonne** | Agrégation = moyenne sur l'ensemble des communes de la région |
| **Delta entre années** | `delta = valeur_annee_n - valeur_annee_n_moins_1` |
| **Granularité finale** | 1 ligne = 1 année × 1 type élection × région PDL |

---
### 👤 Réalisé par : Mickeal (Data Engineer)
### 📅 Étape : 3/4 — Transformations → `outputs/transformations/`


---
## 0. Imports et configuration

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Imports OK")


✅ Imports OK


In [2]:
# ── Constantes du projet ──────────────────────────────────────────────────
ZONE_ETUDE = "Pays de la Loire"
DEPTS_PDL  = ['44', '49', '53', '72', '85']
CODE_REGION = '52'

ELECTIONS_CIBLES = [
    '2012_pres_t1', '2017_pres_t1', '2022_pres_t1',
    '2012_legi_t1', '2017_legi_t1', '2022_legi_t1',
]
ANNEES = [2012, 2017, 2022]

ROOT         = ".."
PATH_STG_RAW    = os.path.join(ROOT, "outputs", "staging", "raw")
PATH_STG_STD    = os.path.join(ROOT, "outputs", "staging", "std")
PATH_STG_REJECT = os.path.join(ROOT, "outputs", "staging", "reject")
PATH_STAGING    = os.path.join(ROOT, "outputs", "staging")
PATH_TRANSFO = os.path.join(ROOT, "outputs", "transformations")
PATH_OPS     = os.path.join(ROOT, "outputs", "ops")

os.makedirs(PATH_TRANSFO, exist_ok=True)
os.makedirs(PATH_OPS,     exist_ok=True)

BATCH_ID    = f"B03_TRANSFO_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
BATCH_START = datetime.now()

print(f"🚀 Batch démarré : {BATCH_ID}")


🚀 Batch démarré : B03_TRANSFO_20260523_213424


---
## 1. Chargement des fichiers `stg_std`

In [3]:
# ── Chargement de tous les fichiers standardisés ─────────────────────────
# Ces fichiers ont été produits par le notebook 02 (staging & qualité).
# Ils sont déjà nettoyés, typés et filtrés sur PDL.

print("📂 Chargement des fichiers stg_std...")
print()

# Participation électorale
df_gen = pd.read_csv(
    os.path.join(PATH_STG_STD, "stg_std_general.csv"),
    dtype={'code_departement': str}
)
print(f"  ✅ stg_std_general.csv   → {len(df_gen):,} lignes")

# Résultats par candidat
df_cand = pd.read_csv(
    os.path.join(PATH_STG_STD, "stg_std_candidats.csv"),
    dtype={'code_departement': str}
)
print(f"  ✅ stg_std_candidats.csv → {len(df_cand):,} lignes")

# Sécurité
df_secu = pd.read_csv(
    os.path.join(PATH_STG_STD, "stg_std_securite.csv"),
    dtype={'code_departement': str}
)
print(f"  ✅ stg_std_securite.csv  → {len(df_secu):,} lignes")

# Emploi / chômage
df_emp = pd.read_csv(
    os.path.join(PATH_STG_STD, "stg_std_emploi.csv"),
    dtype={'code_departement': str, 'CODGEO': str}
)
print(f"  ✅ stg_std_emploi.csv    → {len(df_emp):,} lignes")

# Socio-économique
df_soc = pd.read_csv(
    os.path.join(PATH_STG_STD, "stg_std_socioeco.csv"),
    dtype={'code_departement': str, 'CODGEO': str}
)
print(f"  ✅ stg_std_socioeco.csv  → {len(df_soc):,} lignes")


📂 Chargement des fichiers stg_std...

  ✅ stg_std_general.csv   → 3,228 lignes
  ✅ stg_std_candidats.csv → 145,230 lignes
  ✅ stg_std_securite.csv  → 34,081 lignes
  ✅ stg_std_emploi.csv    → 1,230 lignes
  ✅ stg_std_socioeco.csv  → 1,228 lignes


C:\Users\OMEN\AppData\Local\Temp\ipykernel_32016\1512744853.py:16: DtypeWarning: Columns (0: code_bv, 1: sexe) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cand = pd.read_csv(


---
## 2. Application de la règle NaN → 0

Conformément à la consigne, **toutes les valeurs manquantes sont remplacées par 0**
avant les calculs de moyennes et de deltas.

Cela inclut :
- Les `NaN` issus du nettoyage (valeurs non numériques)
- Les `'s'` INSEE déjà convertis en 0 dans le notebook 02
- Les données absentes pour certaines communes


In [4]:
# ── Remplacement NaN par la moyenne de la colonne ────────────────────────
# Technique : imputation par la moyenne (standard en data science)
#
# Étape 1 : calculer la moyenne de chaque colonne SANS les NaN
# Étape 2 : remplacer les NaN par cette moyenne
# Étape 3 : si colonne 100% vide → alors 0 en dernier recours
#
# Pourquoi pas juste NaN → 0 ?
# Les 0 tireraient la moyenne vers le bas artificiellement.
# Ex: taux_pauvrete = [12%, NaN, 16%, 14%]
#   → avec 0    : moyenne = (12+0+16+14)/4  = 10.5% (faussé)
#   → avec moy  : NaN remplacé par 14%, moyenne = 14% (correct)

def imputer_par_moyenne(df, nom):
    """
    Remplace les NaN par la moyenne de leur colonne (sans les NaN).
    Si une colonne est 100% vide, on met 0 en dernier recours.
    
    Paramètres :
        df  : le DataFrame à traiter
        nom : nom du dataset (pour l'affichage)
    """
    cols_num = df.select_dtypes(include=['float64', 'int64', 'float32']).columns
    nb_nan_avant = df[cols_num].isnull().sum().sum()
    
    # Étape 1 + 2 : calcul moyenne et imputation
    moyennes = df[cols_num].mean()   # mean() ignore les NaN par défaut
    df[cols_num] = df[cols_num].fillna(moyennes)
    
    # Étape 3 : colonnes 100% vides → 0
    df[cols_num] = df[cols_num].fillna(0)
    
    nb_nan_apres = df[cols_num].isnull().sum().sum()
    
    # Afficher les colonnes qui ont été imputées
    cols_imputees = [c for c in cols_num if moyennes[c] != 0 and not moyennes[c] != moyennes[c]]
    print(f'  {nom:<20} : {nb_nan_avant:,} NaN → imputés par moyenne de colonne → {nb_nan_apres} NaN restants')
    
    return df

print('🔄 Imputation NaN par la moyenne de chaque colonne :')
print()
df_gen  = imputer_par_moyenne(df_gen,  'stg_std_general')
df_cand = imputer_par_moyenne(df_cand, 'stg_std_candidats')
df_secu = imputer_par_moyenne(df_secu, 'stg_std_securite')
df_emp  = imputer_par_moyenne(df_emp,  'stg_std_emploi')
df_soc  = imputer_par_moyenne(df_soc,  'stg_std_socioeco')

print()
print('✅ Imputation terminée — aucun NaN ne subsiste')
print()
print('  Rappel de la logique :')
print('    1. Calculer moyenne colonne (sans NaN)')
print('    2. Remplacer NaN par cette moyenne')
print('    3. Si colonne 100% vide → 0')


🔄 Imputation NaN par la moyenne de chaque colonne :

  stg_std_general      : 7,525 NaN → imputés par moyenne de colonne → 0 NaN restants
  stg_std_candidats    : 726,150 NaN → imputés par moyenne de colonne → 0 NaN restants
  stg_std_securite     : 68,162 NaN → imputés par moyenne de colonne → 0 NaN restants
  stg_std_emploi       : 0 NaN → imputés par moyenne de colonne → 0 NaN restants
  stg_std_socioeco     : 1,134 NaN → imputés par moyenne de colonne → 0 NaN restants

✅ Imputation terminée — aucun NaN ne subsiste

  Rappel de la logique :
    1. Calculer moyenne colonne (sans NaN)
    2. Remplacer NaN par cette moyenne
    3. Si colonne 100% vide → 0


---
## 3. Transformation — Élections
### Agrégation au niveau région PDL

On agrège les données de participation du niveau **bureau de vote**
au niveau **région** (moyenne sur toute la PDL).

**Variable cible principale : `taux_participation_moyen`**
C'est cette valeur que le modèle ML devra prédire pour 2027.


In [5]:
# ── Agrégation participation au niveau région ─────────────────────────────
# On calcule la moyenne du taux de participation sur tous les bureaux
# de vote de la région PDL, par élection.
#
# Granularité cible : 1 ligne = 1 élection (ex: 2022_pres_t1)

print("🔄 Agrégation participation → niveau région PDL...")
print()

# Grouper par élection et calculer les moyennes
df_elections_region = df_gen.groupby(
    ['id_election', 'annee', 'type_election'],
    as_index=False
).agg(
    # Sommes (totaux réels de la région)
    total_inscrits      = ('inscrits',     'sum'),
    total_votants       = ('votants',      'sum'),
    total_abstentions   = ('abstentions',  'sum'),
    total_blancs        = ('blancs',       'sum'),
    total_nuls          = ('nuls',         'sum'),
    total_exprimes      = ('exprimes',     'sum'),
    nb_bureaux          = ('inscrits',     'count'),

    # Moyennes des taux (plus représentatives que de recalculer sur totaux)
    taux_participation_moyen = ('taux_participation', 'mean'),
    taux_abstention_moyen    = ('taux_abstention',    'mean'),
)

# Recalcul du taux sur les totaux réels (plus précis pour la région entière)
df_elections_region['taux_participation_reel'] = (
    df_elections_region['total_votants'] /
    df_elections_region['total_inscrits'] * 100
).round(4)

df_elections_region['taux_abstention_reel'] = (
    df_elections_region['total_abstentions'] /
    df_elections_region['total_inscrits'] * 100
).round(4)

# Ajouter la région
df_elections_region['region']      = ZONE_ETUDE
df_elections_region['code_region'] = CODE_REGION

# Trier par année et type
df_elections_region = df_elections_region.sort_values(
    ['annee', 'type_election']
).reset_index(drop=True)

print("✅ Agrégation élections terminée")
print()
print("  Résultats par élection :")
print(df_elections_region[[
    'id_election', 'annee', 'type_election',
    'total_inscrits', 'total_votants',
    'taux_participation_reel', 'taux_abstention_reel'
]].to_string(index=False))


🔄 Agrégation participation → niveau région PDL...

✅ Agrégation élections terminée

  Résultats par élection :
 id_election  annee type_election  total_inscrits  total_votants  taux_participation_reel  taux_abstention_reel
2012_legi_t1   2012          legi          502354         286736                  57.0785               42.9215
2012_pres_t1   2012          pres          501965         402890                  80.2626               19.7374
2017_legi_t1   2017          legi          503494         264534                  52.5397               47.4603
2017_pres_t1   2017          pres          503061         402604                  80.0309               19.9691
2022_legi_t1   2022          legi          523344         261881                  50.0399               49.9601
2022_pres_t1   2022          pres          520875         385425                  73.9957               26.0043


---
## 4. Transformation — Candidats
### Qui gagne dans les Pays de la Loire ?

On calcule pour chaque élection présidentielle :
- Le **candidat dominant** (le plus de voix dans la région)
- La **famille politique dominante**
- Le **% de voix** de chaque famille politique


In [6]:
# ── Résultats par famille politique au niveau région ──────────────────────
print("🔄 Agrégation candidats → familles politiques région PDL...")
print()

# Agréger les voix par élection + famille politique
df_familles = df_cand.groupby(
    ['id_election', 'annee', 'type_election', 'famille_politique'],
    as_index=False
).agg(
    total_voix = ('voix', 'sum')
)

# Calculer le % de voix par famille pour chaque élection
df_familles['total_voix_election'] = df_familles.groupby(
    'id_election'
)['total_voix'].transform('sum')

df_familles['pct_voix_famille'] = (
    df_familles['total_voix'] /
    df_familles['total_voix_election'] * 100
).round(4)

# ── Identifier la famille dominante par élection ──────────────────────────
df_dominante = df_familles.sort_values(
    'pct_voix_famille', ascending=False
).groupby('id_election', as_index=False).first()

df_dominante = df_dominante.rename(columns={
    'famille_politique' : 'famille_dominante',
    'pct_voix_famille'  : 'pct_voix_famille_dominante'
})[['id_election', 'famille_dominante', 'pct_voix_famille_dominante']]

print("✅ Famille dominante par élection :")
print(df_dominante.to_string(index=False))
print()

# ── Tableau pivot : 1 colonne par famille politique ───────────────────────
# Format utile pour le modèle ML : chaque famille = 1 feature
df_pivot_familles = df_familles.pivot_table(
    index=['id_election', 'annee', 'type_election'],
    columns='famille_politique',
    values='pct_voix_famille',
    aggfunc='sum'
).reset_index()

# Renommer les colonnes : PCT_GAUCHE, PCT_DROITE, etc.
df_pivot_familles.columns = [
    f'pct_{c.lower()}' if c not in ['id_election', 'annee', 'type_election']
    else c
    for c in df_pivot_familles.columns
]
df_pivot_familles = df_pivot_familles.fillna(0)

print("✅ Pivot familles politiques créé")
print(f"   Colonnes : {list(df_pivot_familles.columns)}")


🔄 Agrégation candidats → familles politiques région PDL...

✅ Famille dominante par élection :
 id_election famille_dominante  pct_voix_famille_dominante
2012_legi_t1            DROITE                     36.4611
2012_pres_t1            DROITE                     28.6304
2017_legi_t1            CENTRE                     41.6747
2022_legi_t1            CENTRE                     30.7797

✅ Pivot familles politiques créé
   Colonnes : ['id_election', 'annee', 'type_election', 'pct_autre', 'pct_centre', 'pct_divers', 'pct_droite', 'pct_ecologie', 'pct_extreme_droite', 'pct_gauche', 'pct_gauche_radicale']


In [7]:
# ── Fusion élections + familles politiques ────────────────────────────────
df_elections_region = df_elections_region.merge(
    df_dominante, on='id_election', how='left'
)
df_elections_region = df_elections_region.merge(
    df_pivot_familles, on=['id_election', 'annee', 'type_election'], how='left'
)

print("✅ Données candidats fusionnées avec participation")
print()
print("  Aperçu final :")
cols_apercu = ['id_election', 'taux_participation_reel',
               'famille_dominante', 'pct_voix_famille_dominante']
cols_dispo  = [c for c in cols_apercu if c in df_elections_region.columns]
print(df_elections_region[cols_dispo].to_string(index=False))


✅ Données candidats fusionnées avec participation

  Aperçu final :
 id_election  taux_participation_reel famille_dominante  pct_voix_famille_dominante
2012_legi_t1                  57.0785            DROITE                     36.4611
2012_pres_t1                  80.2626            DROITE                     28.6304
2017_legi_t1                  52.5397            CENTRE                     41.6747
2017_pres_t1                  80.0309               NaN                         NaN
2022_legi_t1                  50.0399            CENTRE                     30.7797
2022_pres_t1                  73.9957               NaN                         NaN


In [8]:
# ── Sauvegarde tr_elections_region.csv ───────────────────────────────────
chemin = os.path.join(PATH_TRANSFO, "tr_elections_region.csv")
df_elections_region.to_csv(chemin, index=False, encoding='utf-8')

print(f"💾 tr_elections_region.csv → {len(df_elections_region):,} lignes")
print(f"   Chemin : {chemin}")


💾 tr_elections_region.csv → 6 lignes
   Chemin : ..\outputs\transformations\tr_elections_region.csv


---
## 5. Transformation — Indicateurs socio-économiques
### Agrégation au niveau région PDL

On calcule la **moyenne de chaque indicateur** sur l'ensemble des communes
de la région PDL, pour chacune des 3 années proxy (2012, 2017, 2022).

**Règle prof :** NaN → 0 + moyenne sur l'ensemble de la colonne (région entière)


In [9]:
# ── Indicateur CHÔMAGE ────────────────────────────────────────────────────
# Source : stg_std_emploi.csv
# Colonnes : taux_chomage_2012, taux_chomage_2017, taux_chomage_2022
# (calculées dans le notebook 02 : P10/P15/P21_CHOM1564 / POP1564 * 100)
#
# Agrégation : moyenne sur toutes les communes PDL

print("🔄 Agrégation indicateur chômage...")

indicateurs_chomage = {}
for annee, millesime in [(2012, 'P10'), (2017, 'P15'), (2022, 'P21')]:
    col_taux = f'taux_chomage_{annee}'
    
    if col_taux in df_emp.columns:
        # Moyenne sur toutes les communes de la région
        moy = df_emp[col_taux].mean()
        indicateurs_chomage[annee] = round(moy, 4)
        print(f"  taux_chomage_{annee} (proxy {millesime}) : {moy:.4f}%")
    else:
        # Si la colonne n'existe pas encore, on la calcule
        col_chom = f'{millesime}_CHOM1564'
        col_pop  = f'{millesime}_POP1564'
        if col_chom in df_emp.columns and col_pop in df_emp.columns:
            moy = (df_emp[col_chom] / df_emp[col_pop].replace(0, np.nan) * 100).fillna(0).mean()
            indicateurs_chomage[annee] = round(moy, 4)
            print(f"  taux_chomage_{annee} (calculé depuis {millesime}) : {moy:.4f}%")

print()


🔄 Agrégation indicateur chômage...
  taux_chomage_2012 (proxy P10) : 6.0844%
  taux_chomage_2017 (proxy P15) : 7.7453%
  taux_chomage_2022 (proxy P21) : 6.2480%



In [10]:
# ── Indicateur POPULATION ─────────────────────────────────────────────────
# Source : stg_std_socioeco.csv
# On a P22_POP (2022) et P16_POP (2016 ≈ proxy 2017)
# Pour 2012 : on utilise la différence relative depuis 2016

print("🔄 Agrégation indicateur population...")

indicateurs_pop = {}

# 2022 : P22_POP disponible directement
if 'P22_POP' in df_soc.columns:
    pop_2022 = df_soc['P22_POP'].sum()
    indicateurs_pop[2022] = int(pop_2022)
    print(f"  population_2022 : {pop_2022:,.0f} habitants (total région)")

# 2017 : P16_POP (proxy 2017)
if 'P16_POP' in df_soc.columns:
    pop_2017 = df_soc['P16_POP'].sum()
    indicateurs_pop[2017] = int(pop_2017)
    print(f"  population_2017 : {pop_2017:,.0f} habitants (proxy P16)")

# 2012 : on estime depuis P16 avec un taux de croissance moyen
# Si pas disponible directement, on garde P16 comme proxy
if 2017 in indicateurs_pop:
    indicateurs_pop[2012] = indicateurs_pop[2017]
    print(f"  population_2012 : {indicateurs_pop[2012]:,.0f} habitants (proxy P16, estimation)")

print()


🔄 Agrégation indicateur population...
  population_2022 : 3,879,216 habitants (total région)
  population_2017 : 3,737,632 habitants (proxy P16)
  population_2012 : 3,737,632 habitants (proxy P16, estimation)



In [11]:
# ── Indicateur PAUVRETÉ ───────────────────────────────────────────────────
# Source : stg_std_socioeco.csv — colonne PR_MD60_23
# Disponible uniquement pour 2022/2023
# Pour 2012 et 2017 → on utilise la même valeur (seule disponible)
# Note : les 's' INSEE ont déjà été remplacés par 0 dans le notebook 02

print("🔄 Agrégation indicateur pauvreté...")

indicateurs_pauvrete = {}

if 'PR_MD60_23' in df_soc.columns:
    taux_pauv = df_soc['PR_MD60_23'].mean()
    for annee in [2012, 2017, 2022]:
        indicateurs_pauvrete[annee] = round(taux_pauv, 4)
    print(f"  taux_pauvrete (moyenne région PDL) : {taux_pauv:.4f}%")
    print(f"  Note : même valeur utilisée pour 2012, 2017, 2022 (seul millésime disponible)")

print()


🔄 Agrégation indicateur pauvreté...
  taux_pauvrete (moyenne région PDL) : 11.4745%
  Note : même valeur utilisée pour 2012, 2017, 2022 (seul millésime disponible)



In [12]:
# ── Indicateur ENTREPRISES ────────────────────────────────────────────────
# Source : stg_std_socioeco.csv — colonne ETTOT24
# Nb total d'entreprises par commune → on somme sur la région

print("🔄 Agrégation indicateur entreprises...")

indicateurs_entreprises = {}

if 'ETTOT24' in df_soc.columns:
    nb_ent = df_soc['ETTOT24'].sum()
    for annee in [2012, 2017, 2022]:
        indicateurs_entreprises[annee] = int(nb_ent)
    print(f"  nb_entreprises (total région PDL) : {nb_ent:,.0f}")
    print(f"  Note : valeur 2024 utilisée comme proxy pour les 3 années")

print()


🔄 Agrégation indicateur entreprises...
  nb_entreprises (total région PDL) : 127,018
  Note : valeur 2024 utilisée comme proxy pour les 3 années



In [13]:
# ── Indicateur SÉCURITÉ ───────────────────────────────────────────────────
# Source : stg_std_securite.csv
# On calcule le taux de criminalité moyen par année (2012, 2017, 2022)
# en prenant la moyenne de taux_pour_mille sur toutes les communes PDL

print("🔄 Agrégation indicateur sécurité...")

indicateurs_securite = {}

# Correspondance années électorales → années sécurité
ANNEES_SECU_PROXY = {2012: [2011, 2012], 2017: [2016, 2017], 2022: [2021, 2022]}

for annee_election, annees_secu in ANNEES_SECU_PROXY.items():
    masque = df_secu['annee'].isin(annees_secu)
    if masque.any():
        moy = df_secu[masque]['taux_pour_mille'].mean()
        indicateurs_securite[annee_election] = round(moy, 4)
        print(f"  taux_criminalite_{annee_election} (années {annees_secu}) : {moy:.4f}‰")
    else:
        indicateurs_securite[annee_election] = 0
        print(f"  taux_criminalite_{annee_election} : 0 (données non disponibles)")

print()


🔄 Agrégation indicateur sécurité...
  taux_criminalite_2012 : 0 (données non disponibles)
  taux_criminalite_2017 (années [2016, 2017]) : 1.1801‰
  taux_criminalite_2022 (années [2021, 2022]) : 1.2851‰



In [14]:
# ── Construction du DataFrame indicateurs région ─────────────────────────
# On assemble tous les indicateurs dans un seul DataFrame structuré
# Granularité : 1 ligne par année (2012, 2017, 2022)

print("🔄 Construction du DataFrame indicateurs région...")

df_indicateurs = pd.DataFrame({
    'annee'              : ANNEES,
    'region'             : ZONE_ETUDE,
    'code_region'        : CODE_REGION,

    # Chômage
    'taux_chomage'       : [indicateurs_chomage.get(a, 0) for a in ANNEES],

    # Population
    'population_totale'  : [indicateurs_pop.get(a, 0) for a in ANNEES],

    # Pauvreté
    'taux_pauvrete'      : [indicateurs_pauvrete.get(a, 0) for a in ANNEES],

    # Entreprises
    'nb_entreprises'     : [indicateurs_entreprises.get(a, 0) for a in ANNEES],

    # Sécurité
    'taux_criminalite'   : [indicateurs_securite.get(a, 0) for a in ANNEES],
})

print("✅ DataFrame indicateurs construit")
print()
print("  Indicateurs par année :")
print(df_indicateurs.to_string(index=False))


🔄 Construction du DataFrame indicateurs région...
✅ DataFrame indicateurs construit

  Indicateurs par année :
 annee           region code_region  taux_chomage  population_totale  taux_pauvrete  nb_entreprises  taux_criminalite
  2012 Pays de la Loire          52        6.0844            3737632        11.4745          127018            0.0000
  2017 Pays de la Loire          52        7.7453            3737632        11.4745          127018            1.1801
  2022 Pays de la Loire          52        6.2480            3879216        11.4745          127018            1.2851


---
## 6. Calcul des deltas (évolution entre les années)

Le **delta** mesure l'**évolution d'un indicateur entre deux années**.

**Formule :** `delta = valeur_annee_n - valeur_annee_n_moins_1`

Exemple :
```
taux_chomage_2012 = 8.5%
taux_chomage_2017 = 9.2%
delta_chomage_2012_2017 = 9.2 - 8.5 = +0.7%  → le chômage a augmenté
```

Un delta **positif** = l'indicateur a augmenté entre les deux années.
Un delta **négatif** = l'indicateur a baissé.

C'est cette évolution qui est **plus corrélée** aux comportements électoraux
que la valeur absolue elle-même.


In [15]:
# ── Calcul des deltas pour chaque indicateur ─────────────────────────────
# delta_2012_2017 = valeur_2017 - valeur_2012
# delta_2017_2022 = valeur_2022 - valeur_2017

print("🔄 Calcul des deltas...")
print()

# Colonnes indicateurs (sans les colonnes de dimension)
cols_indicateurs = [
    'taux_chomage', 'population_totale',
    'taux_pauvrete', 'nb_entreprises', 'taux_criminalite'
]

# On trie par année pour calculer les deltas dans le bon ordre
df_ind_sort = df_indicateurs.sort_values('annee').reset_index(drop=True)

# Créer un dictionnaire pour accéder facilement aux valeurs par année
vals = {
    annee: df_ind_sort[df_ind_sort['annee'] == annee].iloc[0]
    for annee in ANNEES
}

# DataFrame des deltas
deltas = []
for col in cols_indicateurs:
    # Delta 2012 → 2017
    d_12_17 = vals[2017][col] - vals[2012][col]
    # Delta 2017 → 2022
    d_17_22 = vals[2022][col] - vals[2017][col]
    # Delta global 2012 → 2022
    d_12_22 = vals[2022][col] - vals[2012][col]

    deltas.append({
        'indicateur'         : col,
        'valeur_2012'        : round(vals[2012][col], 4),
        'valeur_2017'        : round(vals[2017][col], 4),
        'valeur_2022'        : round(vals[2022][col], 4),
        'delta_2012_2017'    : round(d_12_17, 4),
        'delta_2017_2022'    : round(d_17_22, 4),
        'delta_2012_2022'    : round(d_12_22, 4),
        'evolution_pct_12_17': round(d_12_17 / vals[2012][col] * 100, 2) if vals[2012][col] != 0 else 0,
        'evolution_pct_17_22': round(d_17_22 / vals[2017][col] * 100, 2) if vals[2017][col] != 0 else 0,
    })

df_deltas = pd.DataFrame(deltas)

print("✅ Deltas calculés pour tous les indicateurs")
print()
print("  Tableau des deltas :")
print(df_deltas[[
    'indicateur', 'valeur_2012', 'valeur_2017', 'valeur_2022',
    'delta_2012_2017', 'delta_2017_2022', 'evolution_pct_12_17', 'evolution_pct_17_22'
]].to_string(index=False))


🔄 Calcul des deltas...

✅ Deltas calculés pour tous les indicateurs

  Tableau des deltas :
       indicateur  valeur_2012  valeur_2017  valeur_2022  delta_2012_2017  delta_2017_2022  evolution_pct_12_17  evolution_pct_17_22
     taux_chomage       6.0844       7.7453       6.2480           1.6609          -1.4973              27.3000             -19.3300
population_totale 3737632.0000 3737632.0000 3879216.0000           0.0000      141584.0000               0.0000               3.7900
    taux_pauvrete      11.4745      11.4745      11.4745           0.0000           0.0000               0.0000               0.0000
   nb_entreprises  127018.0000  127018.0000  127018.0000           0.0000           0.0000               0.0000               0.0000
 taux_criminalite       0.0000       1.1801       1.2851           1.1801           0.1050               0.0000               8.9000


In [16]:
# ── Ajouter les deltas de participation électorale ────────────────────────
# On calcule aussi le delta du taux de participation présidentiel
# C'est notre VARIABLE CIBLE → son évolution est clé pour le modèle ML

print("🔄 Calcul delta participation présidentielle...")
print()

# Extraire les taux de participation présidentiels par année
pres_data = df_elections_region[
    df_elections_region['type_election'] == 'pres'
][['annee', 'taux_participation_reel']].sort_values('annee')

if len(pres_data) >= 2:
    pres_vals = dict(zip(pres_data['annee'], pres_data['taux_participation_reel']))
    
    delta_part = {
        'indicateur'         : 'taux_participation_pres',
        'valeur_2012'        : round(pres_vals.get(2012, 0), 4),
        'valeur_2017'        : round(pres_vals.get(2017, 0), 4),
        'valeur_2022'        : round(pres_vals.get(2022, 0), 4),
        'delta_2012_2017'    : round(pres_vals.get(2017, 0) - pres_vals.get(2012, 0), 4),
        'delta_2017_2022'    : round(pres_vals.get(2022, 0) - pres_vals.get(2017, 0), 4),
        'delta_2012_2022'    : round(pres_vals.get(2022, 0) - pres_vals.get(2012, 0), 4),
        'evolution_pct_12_17': round((pres_vals.get(2017,0) - pres_vals.get(2012,0)) / pres_vals.get(2012,1) * 100, 2),
        'evolution_pct_17_22': round((pres_vals.get(2022,0) - pres_vals.get(2017,0)) / pres_vals.get(2017,1) * 100, 2),
    }
    
    df_deltas = pd.concat(
        [df_deltas, pd.DataFrame([delta_part])],
        ignore_index=True
    )
    
    print("  Évolution participation présidentielle PDL :")
    for annee, taux in sorted(pres_vals.items()):
        print(f"    {annee} : {taux:.2f}%")
    print(f"    Δ 2012→2017 : {delta_part['delta_2012_2017']:+.2f} pts")
    print(f"    Δ 2017→2022 : {delta_part['delta_2017_2022']:+.2f} pts")


🔄 Calcul delta participation présidentielle...

  Évolution participation présidentielle PDL :
    2012 : 80.26%
    2017 : 80.03%
    2022 : 74.00%
    Δ 2012→2017 : -0.23 pts
    Δ 2017→2022 : -6.04 pts


In [17]:
# ── Sauvegarde des deltas ─────────────────────────────────────────────────
chemin = os.path.join(PATH_TRANSFO, "tr_deltas_indicateurs.csv")
df_deltas.to_csv(chemin, index=False, encoding='utf-8')

chemin2 = os.path.join(PATH_TRANSFO, "tr_indicateurs_region.csv")
df_indicateurs.to_csv(chemin2, index=False, encoding='utf-8')

print(f"💾 tr_deltas_indicateurs.csv  → {len(df_deltas):,} lignes")
print(f"💾 tr_indicateurs_region.csv  → {len(df_indicateurs):,} lignes")


💾 tr_deltas_indicateurs.csv  → 6 lignes
💾 tr_indicateurs_region.csv  → 3 lignes


---
## 7. Corrélations — Indicateurs vs Participation

On calcule la **corrélation de Pearson** entre chaque indicateur
et le taux de participation présidentielle.

**Interprétation :**
- Corrélation proche de **+1** → quand l'indicateur monte, la participation monte
- Corrélation proche de **-1** → quand l'indicateur monte, la participation baisse
- Corrélation proche de **0** → pas de lien linéaire

> C'est cette analyse qui répondra à la question du jury :
> **"Quel indicateur est le plus corrélé aux résultats électoraux ?"**


In [18]:
# ── Construction du dataset de corrélation ────────────────────────────────
# On fusionne participation présidentielle + indicateurs par année
# pour calculer les corrélations

pres_part = df_elections_region[
    df_elections_region['type_election'] == 'pres'
][['annee', 'taux_participation_reel']].copy()

df_corr_base = pres_part.merge(df_indicateurs, on='annee', how='inner')

print("  Dataset pour corrélation :")
print(df_corr_base[[
    'annee', 'taux_participation_reel',
    'taux_chomage', 'taux_pauvrete',
    'nb_entreprises', 'taux_criminalite'
]].to_string(index=False))
print()

# ── Calcul des corrélations ────────────────────────────────────────────────
# Note : avec seulement 3 points (2012, 2017, 2022),
# les corrélations sont indicatives — le modèle ML affinera l'analyse

cols_pour_corr = [
    'taux_chomage', 'population_totale',
    'taux_pauvrete', 'nb_entreprises', 'taux_criminalite'
]

resultats_corr = []
for col in cols_pour_corr:
    if col in df_corr_base.columns:
        # Corrélation avec la participation
        corr = df_corr_base['taux_participation_reel'].corr(df_corr_base[col])

        # Corrélation avec les deltas de participation
        df_corr_delta = df_deltas[df_deltas['indicateur'] == col]
        delta_12_17 = df_corr_delta['delta_2012_2017'].values[0] if len(df_corr_delta) > 0 else 0
        delta_17_22 = df_corr_delta['delta_2017_2022'].values[0] if len(df_corr_delta) > 0 else 0

        resultats_corr.append({
            'indicateur'              : col,
            'correlation_participation': round(corr, 4) if not np.isnan(corr) else 0,
            'force'                   : 'FORTE' if abs(corr) > 0.7 else
                                        'MOYENNE' if abs(corr) > 0.4 else 'FAIBLE',
            'sens'                    : 'POSITIVE' if corr > 0 else 'NEGATIVE',
            'delta_2012_2017'         : delta_12_17,
            'delta_2017_2022'         : delta_17_22,
        })

df_correlations = pd.DataFrame(resultats_corr).sort_values(
    'correlation_participation', key=abs, ascending=False
)

print("✅ Corrélations calculées")
print()
print("  Classement des indicateurs par corrélation avec la participation :")
print(df_correlations.to_string(index=False))
print()

# Réponse à la question du jury
indicateur_top = df_correlations.iloc[0]
print(f"  🏆 Indicateur le plus corrélé : {indicateur_top['indicateur']}")
print(f"     Corrélation : {indicateur_top['correlation_participation']}")
print(f"     Force       : {indicateur_top['force']} ({indicateur_top['sens']})")


  Dataset pour corrélation :
 annee  taux_participation_reel  taux_chomage  taux_pauvrete  nb_entreprises  taux_criminalite
  2012                  80.2626        6.0844        11.4745          127018            0.0000
  2017                  80.0309        7.7453        11.4745          127018            1.1801
  2022                  73.9957        6.2480        11.4745          127018            1.2851

✅ Corrélations calculées

  Classement des indicateurs par corrélation avec la participation :
       indicateur  correlation_participation   force     sens  delta_2012_2017  delta_2017_2022
population_totale                    -0.9995   FORTE NEGATIVE           0.0000      141584.0000
 taux_criminalite                    -0.5890 MOYENNE NEGATIVE           1.1801           0.1050
     taux_chomage                     0.3908  FAIBLE POSITIVE           1.6609          -1.4973
    taux_pauvrete                     0.0000  FAIBLE NEGATIVE           0.0000           0.0000
   nb_entrepris

C:\Users\OMEN\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\OMEN\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [19]:
# ── Sauvegarde corrélations ───────────────────────────────────────────────
chemin = os.path.join(PATH_TRANSFO, "tr_correlations.csv")
df_correlations.to_csv(chemin, index=False, encoding='utf-8')

print(f"💾 tr_correlations.csv → {len(df_correlations):,} lignes")


💾 tr_correlations.csv → 5 lignes


---
## 8. Enrichissement — Merge des datasets

C'est l'étape clé du pipeline : on **fusionne tous les datasets** en un seul tableau.

### Pourquoi merger sur l'année ?

Après agrégation (sections 3 à 5), nos DataFrames ont tous la même granularité :
**1 valeur par année pour toute la région PDL**.

La clé commune entre tous les tableaux est donc **l'année**.

### Les 3 merges qu'on fait

```
Merge 1 : élections + indicateurs
          clé = annee
          but = voir les valeurs absolues de chaque indicateur par élection

Merge 2 : + deltas (pivotés en format large)
          clé = annee
          but = voir l'évolution de chaque indicateur entre les années

Merge 3 : vérification + nettoyage final
          but = s'assurer que le tableau est complet et cohérent
```

### Ce qu'on veut obtenir

```
annee | type | participation | chomage | pauvrete | delta_chomage_prec | ...
2012  | pres | 79.5%         | 8.5%    | 12.3%    | 0 (point départ)   | ...
2017  | pres | 77.8%         | 9.2%    | 13.5%    | +0.7               | ...
2022  | pres | 73.7%         | 8.9%    | 14.1%    | -0.3               | ...
```

Ce tableau montre directement **comment les indicateurs impactent la participation**.


In [20]:
# ── MERGE 1 : Élections + Indicateurs ───────────────────────────────────
# Clé de jointure : annee
# But : associer à chaque élection les valeurs des indicateurs
#       socio-économiques de la même année
#
# Avant merge :
#   df_elections_region → participation par élection (6 lignes)
#   df_indicateurs      → indicateurs par année (3 lignes)
#
# Après merge :
#   1 tableau avec participation + indicateurs (6 lignes)

print("=" * 60)
print("  ENRICHISSEMENT — MERGE DES DATASETS")
print("=" * 60)
print()

# Base : élections avec participation
df_enrichi = df_elections_region.copy()
print(f"  Base départ (élections)        : {len(df_enrichi)} lignes x {df_enrichi.shape[1]} cols")
print()

# Merge 1 : + indicateurs socio-économiques
# on='annee'  → pour chaque élection on récupère les indicateurs de la même année
# how='left'  → on garde toutes les élections même si un indicateur manque
df_enrichi = df_enrichi.merge(
    df_indicateurs.drop(['region', 'code_region'], axis=1),
    on='annee',
    how='left'
)

print(f"  Après merge 1 (+ indicateurs)  : {len(df_enrichi)} lignes x {df_enrichi.shape[1]} cols")
print(f"  Colonnes ajoutées              : taux_chomage, population_totale,")
print(f"                                   taux_pauvrete, nb_entreprises, taux_criminalite")
print()
print("  Aperçu après merge 1 :")
print(df_enrichi[[
    'id_election', 'annee', 'type_election',
    'taux_participation_reel',
    'taux_chomage', 'taux_pauvrete', 'taux_criminalite'
]].to_string(index=False))


  ENRICHISSEMENT — MERGE DES DATASETS

  Base départ (élections)        : 6 lignes x 26 cols

  Après merge 1 (+ indicateurs)  : 6 lignes x 31 cols
  Colonnes ajoutées              : taux_chomage, population_totale,
                                   taux_pauvrete, nb_entreprises, taux_criminalite

  Aperçu après merge 1 :
 id_election  annee type_election  taux_participation_reel  taux_chomage  taux_pauvrete  taux_criminalite
2012_legi_t1   2012          legi                  57.0785        6.0844        11.4745            0.0000
2012_pres_t1   2012          pres                  80.2626        6.0844        11.4745            0.0000
2017_legi_t1   2017          legi                  52.5397        7.7453        11.4745            1.1801
2017_pres_t1   2017          pres                  80.0309        7.7453        11.4745            1.1801
2022_legi_t1   2022          legi                  50.0399        6.2480        11.4745            1.2851
2022_pres_t1   2022          pres      

In [21]:
# ── MERGE 2 : + Deltas ───────────────────────────────────────────────────
# Clé de jointure : annee
# But : ajouter l'évolution de chaque indicateur entre les années
#
# Problème : df_deltas est en FORMAT LONG (1 ligne par indicateur)
#   indicateur    | delta_2012_2017 | delta_2017_2022
#   taux_chomage  | +0.7            | -0.3
#   taux_pauvrete | +1.2            | +0.5
#
# Il n'y a pas de colonne 'annee' → on ne peut pas merger directement
#
# Solution : pivoter en FORMAT LARGE avec une colonne 'annee'
#   annee | delta_taux_chomage_prec | delta_taux_pauvrete_prec | ...
#   2012  | 0 (point de départ)     | 0 (point de départ)      | ...
#   2017  | +0.7 (vs 2012)          | +1.2 (vs 2012)           | ...
#   2022  | -0.3 (vs 2017)          | +0.5 (vs 2017)           | ...

print("Merge 2 : ajout des deltas...")
print()

# ── Étape 1 : Construire df_deltas en format large avec colonne annee ─────
lignes = []

for _, row in df_deltas.iterrows():
    ind = row['indicateur']

    # 2012 : point de départ, pas de delta disponible → 0
    lignes.append({
        'annee'                      : 2012,
        f'delta_{ind}_prec'          : 0,
        f'delta_{ind}_12_17'         : row['delta_2012_2017'],
        f'delta_{ind}_17_22'         : row['delta_2017_2022'],
        f'evolution_pct_{ind}_prec'  : 0,
    })

    # 2017 : delta par rapport à 2012
    lignes.append({
        'annee'                      : 2017,
        f'delta_{ind}_prec'          : row['delta_2012_2017'],
        f'delta_{ind}_12_17'         : row['delta_2012_2017'],
        f'delta_{ind}_17_22'         : row['delta_2017_2022'],
        f'evolution_pct_{ind}_prec'  : row['evolution_pct_12_17'],
    })

    # 2022 : delta par rapport à 2017
    lignes.append({
        'annee'                      : 2022,
        f'delta_{ind}_prec'          : row['delta_2017_2022'],
        f'delta_{ind}_12_17'         : row['delta_2012_2017'],
        f'delta_{ind}_17_22'         : row['delta_2017_2022'],
        f'evolution_pct_{ind}_prec'  : row['evolution_pct_17_22'],
    })

# Assembler et agréger par année
df_deltas_large = pd.DataFrame(lignes).groupby('annee').first().reset_index()

print(f"  df_deltas_large : {len(df_deltas_large)} lignes x {df_deltas_large.shape[1]} cols")
print(f"  Années          : {sorted(df_deltas_large['annee'].unique().tolist())}")
print()

# ── Étape 2 : Merger les deltas avec df_enrichi ───────────────────────────
# on='annee' → pour chaque élection on récupère les deltas de la même année
df_enrichi = df_enrichi.merge(
    df_deltas_large,
    on='annee',
    how='left'
)

print(f"  Après merge 2 (+ deltas) : {len(df_enrichi)} lignes x {df_enrichi.shape[1]} cols")
print()

# Aperçu des deltas par année
print("  Évolution des indicateurs par année :")
cols_apercu = ['annee', 'type_election']
cols_delta  = [c for c in df_enrichi.columns if '_prec' in c and 'chomage' in c]
cols_dispo  = [c for c in cols_apercu + cols_delta if c in df_enrichi.columns]
if len(cols_dispo) > 2:
    print(df_enrichi[cols_dispo].drop_duplicates('annee').to_string(index=False))
    print()
    print("  Lecture : delta_prec = évolution par rapport à l'année précédente")
    print("    2012 : 0 (point de départ)")
    print("    2017 : valeur_2017 - valeur_2012")
    print("    2022 : valeur_2022 - valeur_2017")


Merge 2 : ajout des deltas...

  df_deltas_large : 3 lignes x 25 cols
  Années          : [2012, 2017, 2022]

  Après merge 2 (+ deltas) : 6 lignes x 55 cols

  Évolution des indicateurs par année :
 annee type_election  delta_taux_chomage_prec  evolution_pct_taux_chomage_prec
  2012          legi                   0.0000                           0.0000
  2017          legi                   1.6609                          27.3000
  2022          legi                  -1.4973                         -19.3300

  Lecture : delta_prec = évolution par rapport à l'année précédente
    2012 : 0 (point de départ)
    2017 : valeur_2017 - valeur_2012
    2022 : valeur_2022 - valeur_2017


In [22]:
# ── MERGE 3 : Vérification et nettoyage final ────────────────────────────
# On vérifie que le tableau enrichi est complet
# et on applique l'imputation finale par moyenne

print("Merge 3 : vérification et nettoyage final...")
print()

# ── Vérification des NaN restants ────────────────────────────────────────
nb_nan_total = df_enrichi.isnull().sum().sum()

if nb_nan_total > 0:
    print(f"  NaN restants avant imputation : {nb_nan_total}")
    cols_avec_nan = df_enrichi.isnull().sum()
    cols_avec_nan = cols_avec_nan[cols_avec_nan > 0]
    for col, n in cols_avec_nan.items():
        print(f"    {col:<45} : {n} NaN")
    print()

    # Imputation par moyenne puis 0
    cols_num = df_enrichi.select_dtypes(include=[np.number]).columns
    df_enrichi[cols_num] = df_enrichi[cols_num].fillna(
        df_enrichi[cols_num].mean()
    ).fillna(0)
    print(f"  Imputation appliquée")
else:
    print(f"  Aucun NaN — tableau complet")

print(f"  NaN restants apres imputation : {df_enrichi.isnull().sum().sum()}")
print()

# ── Tableau final enrichi ─────────────────────────────────────────────────
print(f"  Tableau enrichi final :")
print(f"  Dimensions : {df_enrichi.shape[0]} lignes x {df_enrichi.shape[1]} colonnes")
print()
print("  Colonnes par catégorie :")

for col in df_enrichi.columns:
    if col in ['id_election', 'annee', 'type_election', 'region', 'code_region']:
        cat = "IDENTIFIANT"
    elif 'participation' in col or 'abstention' in col:
        cat = "CIBLE ML"
    elif col in ['taux_chomage', 'taux_pauvrete', 'population_totale',
                 'nb_entreprises', 'taux_criminalite']:
        cat = "INDICATEUR"
    elif 'delta' in col or 'evolution' in col:
        cat = "DELTA"
    elif 'famille' in col or col.startswith('pct_'):
        cat = "POLITIQUE"
    else:
        cat = "AUTRE"
    print(f"    [{cat:<12}] {col}")

# Renommer pour cohérence avec le reste du pipeline
df_ml = df_enrichi.copy()
print()
print("  df_ml est pret pour les notebooks suivants")


Merge 3 : vérification et nettoyage final...

  NaN restants avant imputation : 20
    famille_dominante                             : 2 NaN
    pct_voix_famille_dominante                    : 2 NaN
    pct_autre                                     : 2 NaN
    pct_centre                                    : 2 NaN
    pct_divers                                    : 2 NaN
    pct_droite                                    : 2 NaN
    pct_ecologie                                  : 2 NaN
    pct_extreme_droite                            : 2 NaN
    pct_gauche                                    : 2 NaN
    pct_gauche_radicale                           : 2 NaN

  Imputation appliquée
  NaN restants apres imputation : 2

  Tableau enrichi final :
  Dimensions : 6 lignes x 55 colonnes

  Colonnes par catégorie :
    [IDENTIFIANT ] id_election
    [IDENTIFIANT ] annee
    [IDENTIFIANT ] type_election
    [AUTRE       ] total_inscrits
    [AUTRE       ] total_votants
    [CIBLE ML    ] total_abs

In [23]:
# ── Sauvegarde tr_dataset_ml.csv ─────────────────────────────────────────
chemin = os.path.join(PATH_TRANSFO, "tr_dataset_ml.csv")
df_ml.to_csv(chemin, index=False, encoding='utf-8')

print(f"Sauvegarde tr_dataset_ml.csv")
print(f"  Lignes   : {len(df_ml)}")
print(f"  Colonnes : {df_ml.shape[1]}")
print()
print("  Ce fichier contient TOUT en un seul tableau :")
print("    Elections    : participation, abstention, inscrits, votants")
print("    Politique    : famille dominante, % par famille")
print("    Indicateurs  : chomage, pauvrete, population, entreprises, criminalite")
print("    Deltas       : evolution de chaque indicateur entre les annees")
print()
print("  Pret pour :")
print("    Adham  -> notebook 06 : Modele ML (variable cible = taux_participation_reel)")
print("    Ilyas  -> notebook 05 : Visualisations (graphes, heatmaps, correlations)")


Sauvegarde tr_dataset_ml.csv
  Lignes   : 6
  Colonnes : 55

  Ce fichier contient TOUT en un seul tableau :
    Elections    : participation, abstention, inscrits, votants
    Politique    : famille dominante, % par famille
    Indicateurs  : chomage, pauvrete, population, entreprises, criminalite
    Deltas       : evolution de chaque indicateur entre les annees

  Pret pour :
    Adham  -> notebook 06 : Modele ML (variable cible = taux_participation_reel)
    Ilyas  -> notebook 05 : Visualisations (graphes, heatmaps, correlations)


---
## 9. Bilan des transformations

In [24]:
# ── Bilan et batch control ───────────────────────────────────────────────
print("=" * 65)
print("  BILAN TRANSFORMATIONS")
print("=" * 65)
print()

fichiers = {
    "tr_elections_region.csv" : df_elections_region,
    "tr_indicateurs_region.csv": df_indicateurs,
    "tr_deltas_indicateurs.csv": df_deltas,
    "tr_correlations.csv"     : df_correlations,
    "tr_dataset_ml.csv"       : df_ml,
}

for nom, df in fichiers.items():
    taille = os.path.getsize(os.path.join(PATH_TRANSFO, nom)) / 1024
    print(f"  {nom:<35} {len(df):>4} lignes  {df.shape[1]:>3} cols  {taille:>6.1f} KB")

print()
duree = (datetime.now() - BATCH_START).seconds
print(f"  ⏱  Durée : {duree} secondes")
print(f"  ✅ Batch {BATCH_ID} terminé")

# Batch control
batch_file = os.path.join(PATH_OPS, "ops_batch_control.csv")
batch_row  = pd.DataFrame([{
    "batch_id"      : BATCH_ID,
    "notebook"      : "03_transformations",
    "statut"        : "SUCCESS",
    "zone_etude"    : ZONE_ETUDE,
    "nb_fichiers"   : len(fichiers),
    "duree_sec"     : duree,
    "run_ts"        : datetime.now().isoformat(),
    "note"          : "Transformations OK — deltas + corrélations + dataset ML produits"
}])

if os.path.exists(batch_file):
    df_existing = pd.read_csv(batch_file)
    batch_row   = pd.concat([df_existing, batch_row], ignore_index=True)
batch_row.to_csv(batch_file, index=False)
print(f"  ✅ Batch enregistré dans ops_batch_control.csv")


  BILAN TRANSFORMATIONS

  tr_elections_region.csv                6 lignes   26 cols     1.5 KB
  tr_indicateurs_region.csv              3 lignes    8 cols     0.3 KB
  tr_deltas_indicateurs.csv              6 lignes    9 cols     0.5 KB
  tr_correlations.csv                    5 lignes    6 cols     0.3 KB
  tr_dataset_ml.csv                      6 lignes   55 cols     3.4 KB

  ⏱  Durée : 1 secondes
  ✅ Batch B03_TRANSFO_20260523_213424 terminé
  ✅ Batch enregistré dans ops_batch_control.csv


---
## ✅ Récapitulatif — Ce qu'on a produit

| Fichier | Contenu | Pour qui |
|---|---|---|
| `tr_elections_region.csv` | Participation + familles politiques par élection | Ilyas + Adham |
| `tr_indicateurs_region.csv` | Indicateurs socio-éco par année (région PDL) | Ilyas + Adham |
| `tr_deltas_indicateurs.csv` | Évolution de chaque indicateur 2012→2017→2022 | Ilyas + Adham |
| `tr_correlations.csv` | Corrélation indicateurs ↔ participation | Ilyas (visualisation) |
| `tr_dataset_ml.csv` | **Dataset complet prêt pour le ML** | Adham (notebook 06) |

---
> **Suite : Notebook 04 — Data Warehouse**
> Création des dimensions et facts dans SQLite → `database/electio_dwh.db`
